# Repaso desde cero: jugar con reglas y programar un CSP

Imagina que tienes juguetes, cajitas y unas reglas para ordenarlos.
Tú explicas las reglas. La computadora busca una forma de ordenar todo sin romperlas.
Eso es lo que aprenderemos a programar.

**Puedes empezar sin saber Python ni lógica.** Primero habrá una historia pequeña;
después veremos cómo escribirla. Las palabras del curso aparecerán junto a su significado.

Este repaso se preparó leyendo los **9 notebooks del repositorio**. Al final están
sus nombres y qué ideas aporta cada uno. Usaremos ejemplos pequeños para aprender
los temas de CSP; también conectaremos con tus médicos, proyectos, horarios y obreros.

## Cómo usar este cuaderno

1. Lee una explicación y trata de adivinar qué ocurrirá.
2. Ejecuta su celda de código con **Shift + Enter**.
3. Mira la respuesta y compárala con las reglas.
4. Cambia un número cuando la actividad lo indique y vuelve a ejecutar esa celda.

Ejecuta en orden, de arriba hacia abajo. Los modelos principales empiezan de nuevo
para que sus reglas no se mezclen. Las importaciones y algunas herramientas se reutilizan.

## Nuestro camino

| Parada | Qué aprenderás |
|---|---|
| 1–4 | Cajitas, reglas, Python mínimo y un primer programa completo |
| 5–8 | Booleanas, ALO, AMO, EO, Y, O, NO e implicación |
| 9–11 | Reificación, `b.Not()` y tu pregunta 4 completa |
| 12–16 | Turnos, listas de listas, horarios, grupos y contar coincidencias |
| 17–21 | Productos, tablas, colores, cuadrados, letras y palabras |
| 22–24 | Tiempo, intervalos, capacidad y dos formas de programar la obra |
| 25–28 | Optimización, varias soluciones, errores y ejercicios con respuestas |

Puedes leer una parada por vez. No necesitas memorizar todos los nombres hoy.

## 1. El juego de las cajitas: ¿qué es un CSP?

Ana y Beto quieren un juguete cada uno. Hay un oso y un tren.
La regla es: **no pueden elegir el mismo juguete**.

| Palabra del curso | En nuestro juego |
|---|---|
| **Variable** | Una cajita que guardará qué juguete elige Ana o Beto |
| **Dominio** | Los valores que pueden entrar en esa cajita: oso o tren |
| **Restricción** | Una regla: sus juguetes deben ser distintos |
| **Solución** | Una elección que cumple todas las reglas |

Como CP-SAT trabaja con números enteros, acordamos: `1 = oso`, `2 = tren`.

```text
                  OPCIONES                 REGLA
Ana  [ ? ]       {1: oso, 2: tren}       Ana ≠ Beto
Beto [ ? ]       {1: oso, 2: tren}

Una solución: Ana [1: oso] y Beto [2: tren]
Otra solución: Ana [2: tren] y Beto [1: oso]
```

**CSP** viene de *Constraint Satisfaction Problem*: problema de satisfacción de
restricciones. Formalmente se escribe **(X, D, C)**: variables, dominios y restricciones.
Aquí `X = {Ana, Beto}`, cada dominio es `{1, 2}` y `C = {Ana ≠ Beto}`.

El **modelo** es la hoja donde anotamos el juego. El **solver** es el ayudante que
busca una solución. Escribir las reglas y buscar la respuesta son pasos diferentes.

## 2. Python mínimo para entender lo que vas a escribir

| Código | Cómo leerlo |
|---|---|
| `cantidad = 3` | Guarda el número 3 con ese nombre |
| `x == 3` | Pregunta o expresa que x es igual a 3 |
| `x != 3` | x es distinto de 3 |
| `x < y`, `x > y` | x es menor / mayor que y |
| `x <= y`, `x >= y` | Menor o igual / mayor o igual |
| `[1, 0, 1]` | Una lista: tres cosas ordenadas |
| `sum([1, 0, 1])` | Suma los números: da 2 |
| `range(3)` | Recorre 0, 1 y 2; el 3 queda fuera |
| `# explicación` | Un comentario para quien lee; Python no lo ejecuta |

**`=` y `==` son distintos.** `nombre = ...` pone un nombre; `==` expresa igualdad.

Las posiciones de una lista empiezan en **0**: en `juguetes`, la posición 0 es
el primer juguete. Un `for` significa «repite esto para cada elemento».

In [1]:
juguetes = ["oso", "tren", "pelota"]
print("Primer juguete:", juguetes[0])
print("Segundo juguete:", juguetes[1])
print("Suma:", sum([1, 0, 1]))

for i in range(3):
    print("Posición", i, "->", juguetes[i])

Primer juguete: oso
Segundo juguete: tren
Suma: 2
Posición 0 -> oso
Posición 1 -> tren
Posición 2 -> pelota


## 3. Preparar la herramienta

**OR-Tools** es la biblioteca; **CP-SAT** es el solver que usaremos.
La celda siguiente instala la versión usada en el repaso anterior del repositorio.
Necesita internet la primera vez. Si Jupyter o Colab pide reiniciar el kernel,
hazlo y sigue desde la importación.

Usaremos `NewIntVar`, `Add`, `OnlyEnforceIf`, etc., para que reconozcas los nombres
de tus evaluaciones. La documentación también muestra `new_int_var`, `add` y
`only_enforce_if`: son los nombres equivalentes en esta versión.

In [ ]:
%pip install -q ortools==9.14.6206

In [2]:
from ortools.sat.python import cp_model
import ortools

print("OR-Tools:", ortools.__version__)

OR-Tools: 9.14.6206


## 4. Tu primer CSP completo: repartir dos juguetes

La receta tiene cinco pasos: **crear modelo → crear variables → agregar reglas →
resolver → leer la respuesta**. Aquí se ven todos, sin esconder la resolución.

In [3]:
# 1. La hoja de reglas.
model = cp_model.CpModel()

# 2. Dos decisiones. Los límites 1 y 2 SÍ están incluidos.
ana = model.NewIntVar(1, 2, "juguete_de_ana")
beto = model.NewIntVar(1, 2, "juguete_de_beto")

# 3. No pueden tener el mismo juguete.
model.Add(ana != beto)

# 4. Crear el ayudante y pedirle que busque.
solver = cp_model.CpSolver()
status = solver.Solve(model)
print("Estado:", solver.StatusName(status))

# 5. Leer los valores únicamente si existe una solución.
if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    nombres = {1: "oso", 2: "tren"}
    print("Ana:", nombres[solver.Value(ana)])
    print("Beto:", nombres[solver.Value(beto)])
    print("Tiempo:", solver.WallTime(), "segundos")
else:
    print("No hay una solución disponible para mostrar.")

Estado: OPTIMAL
Ana: oso
Beto: tren
Tiempo: 0.007895300000000001 segundos


Puede salir Ana con oso o Ana con tren: **las dos respuestas son válidas**.
No pedimos una respuesta particular. `solver.Value(ana)` lee la elección que
encontró el solver; `ana` por sí sola es la variable del modelo.

Una variable del modelo todavía no es un número conocido. `model.Add(ana != beto)`
anota una regla; no elige inmediatamente los juguetes.

| Estado del solver | Qué sabemos |
|---|---|
| `OPTIMAL` | Encontró solución; si había objetivo, demostró que es la mejor |
| `FEASIBLE` | Encontró solución; podría faltar demostrar que es la mejor |
| `INFEASIBLE` | Demostró que ninguna combinación cumple todas las reglas |
| `UNKNOWN` | Todavía no pudo concluir; por ejemplo, se terminó el tiempo |
| `MODEL_INVALID` | Hay un problema en cómo se construyó el modelo |

`UNKNOWN` **no significa** que sea imposible. `OPTIMAL` **no significa** que la
solución sea única ni que se hayan mostrado todas.

En las siguientes paradas repetiremos estos pasos. Esta pequeña función solo
ahorra las líneas de crear y ejecutar el solver; también permite leer el estado.

In [4]:
def resolver(modelo):
    error = modelo.Validate()  # Revisa la estructura; no demuestra que haya solución.
    if error:
        raise ValueError(error)
    solucionador = cp_model.CpSolver()
    solucionador.parameters.max_time_in_seconds = 10
    solucionador.parameters.num_search_workers = 1
    estado = solucionador.Solve(modelo)
    print("Estado:", solucionador.StatusName(estado))
    return solucionador, estado


HAY_SOLUCION = (cp_model.OPTIMAL, cp_model.FEASIBLE)

## 5. Booleanas: una lucecita que dice SÍ o NO

Una booleana tiene dos valores: **1 = sí / verdadero** y **0 = no / falso**.
Por ejemplo, `elijo_oso` dice si elegimos el oso.

```text
elijo_oso = 1     [encendida]   Sí lo elegimos.
elijo_oso = 0     [apagada]     No lo elegimos.
```

`model.NewBoolVar("elijo_oso")` crea esa decisión. No significa que ya valga 1:
el solver elegirá su valor según las reglas.

Hay dos usos que veremos: **decidir** si elegimos algo y **señalar** si otra condición
es cierta. En el segundo uso hay que conectar la lucecita con la condición.

Lo útil de 0 y 1 es que podemos sumarlos para **contar cuántos sí hay**:
`1 + 0 + 1 = 2` juguetes elegidos.

In [5]:
model = cp_model.CpModel()
oso = model.NewBoolVar("elijo_oso")
tren = model.NewBoolVar("elijo_tren")
pelota = model.NewBoolVar("elijo_pelota")

model.Add(oso == 1)
model.Add(tren == 0)
model.Add(pelota == 1)

solver, status = resolver(model)
if status in HAY_SOLUCION:
    valores = [solver.Value(oso), solver.Value(tren), solver.Value(pelota)]
    print("Oso, tren, pelota:", valores)
    print("Cantidad elegida:", sum(valores))  # 2

Estado: OPTIMAL
Oso, tren, pelota: [1, 0, 1]
Cantidad elegida: 2


## 6. ALO, AMO y EO: cuántas lucecitas pueden estar encendidas

Piensa en llevar juguetes al parque.

| Nombre | Qué te dicen | Cantidades permitidas con 3 juguetes | Ecuación |
|---|---|---|---|
| **ALO** (*At Least One*) | «Lleva al menos uno» | 1, 2 o 3 | `sum(elegidos) >= 1` |
| **AMO** (*At Most One*) | «Puedes llevar como máximo uno» | 0 o 1 | `sum(elegidos) <= 1` |
| **EO** (*Exactly One*) | «Lleva exactamente uno» | Solo 1 | `sum(elegidos) == 1` |

**AMO permite no llevar nada. ALO permite llevar varios. EO exige uno y solo uno.**
También puedes ver el nombre **EXO** para exactamente uno.

Cada fila siguiente es una elección completa. `1` significa llevar el juguete.

| Oso | Tren | Pelota | Cantidad | ¿ALO? | ¿AMO? | ¿EO? |
|---|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 | No | Sí | No |
| 0 | 0 | 1 | 1 | Sí | Sí | Sí |
| 0 | 1 | 0 | 1 | Sí | Sí | Sí |
| 0 | 1 | 1 | 2 | Sí | No | No |
| 1 | 0 | 0 | 1 | Sí | Sí | Sí |
| 1 | 0 | 1 | 2 | Sí | No | No |
| 1 | 1 | 0 | 2 | Sí | No | No |
| 1 | 1 | 1 | 3 | Sí | No | No |

EO combina las dos reglas: **al menos uno y a lo sumo uno**.

En OR-Tools estas parejas de instrucciones expresan lo mismo. Elige una de las
dos formas de cada fila; no hace falta escribir las dos.

| Regla | Con suma | Con método específico |
|---|---|---|
| ALO | `model.Add(sum(b) >= 1)` | `model.AddBoolOr(b)` |
| AMO | `model.Add(sum(b) <= 1)` | `model.AddAtMostOne(b)` |
| EO | `model.Add(sum(b) == 1)` | `model.AddExactlyOne(b)` |

Aquí probamos las tres reglas con la misma historia. Mira que el modelo se crea
de nuevo en cada vuelta. El `if` de Python elige qué regla programar según un
**texto conocido**; no pregunta por una decisión pendiente del solver.

In [6]:
for regla in ["ALO", "AMO", "EO"]:
    print("\nRegla:", regla)
    model = cp_model.CpModel()
    oso = model.NewBoolVar("oso")
    tren = model.NewBoolVar("tren")
    pelota = model.NewBoolVar("pelota")
    elegidos = [oso, tren, pelota]

    if regla == "ALO":
        model.AddBoolOr(elegidos)
    elif regla == "AMO":
        model.AddAtMostOne(elegidos)
    else:
        model.AddExactlyOne(elegidos)

    solver, status = resolver(model)
    if status in HAY_SOLUCION:
        valores = [solver.Value(b) for b in elegidos]
        print("Oso, tren, pelota:", valores, "Cantidad:", sum(valores))


Regla: ALO
Estado: OPTIMAL
Oso, tren, pelota: [1, 0, 0] Cantidad: 1

Regla: AMO
Estado: OPTIMAL
Oso, tren, pelota: [0, 0, 0] Cantidad: 0

Regla: EO
Estado: OPTIMAL
Oso, tren, pelota: [1, 0, 0] Cantidad: 1


Un solver que aplica AMO puede devolver un juguete o ninguno. Un ejemplo concreto
no muestra todas las opciones que permite una regla; la tabla sí.

**De uno a k:** para «al menos 2», escribe `model.Add(sum(b) >= 2)`;
para «como máximo 2», `model.Add(sum(b) <= 2)`; para «exactamente 2»,
`model.Add(sum(b) == 2)`. Estas sumas cuentan elecciones porque `b` contiene
booleanas. Si contiene días como `[3, 4, 5]`, su suma no cuenta cuántos días hay.

## 7. NO, Y, O y XOR

| Operación | Con dos lucecitas | Cómo exigirla en OR-Tools |
|---|---|---|
| **NO / NOT** | Invierte sí y no | `b.Not()` representa la negación |
| **Y / AND** | Ambas deben ser 1 | `model.AddBoolAnd([a, b])` |
| **O / OR** | Al menos una debe ser 1 | `model.AddBoolOr([a, b])` |
| **XOR** | Exactamente una de las dos debe ser 1 | `model.AddBoolXOr([a, b])` |

```text
b       0   1
b.Not() 1   0
```

`b.Not()` no cambia el valor de `b` ni crea otra elección independiente: representa
«b es falso». También verás `~b` como forma equivalente. No uses `not b` de Python
para construir esta negación del modelo.

El O lógico permite que **ambas** sean verdaderas. Con **tres o más** booleanas,
`AddBoolXOr` exige una cantidad **impar** de verdaderas: con tres permite 1 **o 3**.
Por eso, para elegir exactamente un juguete entre muchos, usa `AddExactlyOne`.

In [7]:
model = cp_model.CpModel()
casco = model.NewBoolVar("llevo_casco")
bicicleta = model.NewBoolVar("llevo_bicicleta")

model.AddBoolAnd([casco, bicicleta])  # Las dos deben valer 1.

solver, status = resolver(model)
if status in HAY_SOLUCION:
    print("Casco:", solver.Value(casco), "Bicicleta:", solver.Value(bicicleta))
    print("NO casco:", solver.Value(casco.Not()))

Estado: OPTIMAL
Casco: 1 Bicicleta: 1
NO casco: 0


## 8. «Si…, entonces…»: una promesa

«**Si llevo bicicleta, entonces llevo casco**».
La condición es llevar bicicleta. La consecuencia es llevar casco.
También se llaman **antecedente** y **consecuente**.

| Bicicleta | Casco | ¿Se cumple la promesa? |
|---|---|---|
| 0 | 0 | Sí: no llevé bicicleta |
| 0 | 1 | Sí: puedo llevar casco aunque no lleve bicicleta |
| 1 | 0 | **No: llevé bicicleta sin casco** |
| 1 | 1 | Sí: llevé bicicleta con casco |

La flecha `→` significa «si lo primero ocurre, exige lo segundo».
No obliga a que lo primero ocurra. Tampoco permite deducir que hay bicicleta solo
porque hay casco.

Entre dos booleanas usamos `AddImplication`. Para activar una restricción sobre
números usamos `OnlyEnforceIf`.

In [8]:
model = cp_model.CpModel()
bicicleta = model.NewBoolVar("bicicleta")
casco = model.NewBoolVar("casco")

model.AddImplication(bicicleta, casco)  # bicicleta -> casco
model.Add(bicicleta == 1)             # En este ejemplo sí llevamos bicicleta.

solver, status = resolver(model)
if status in HAY_SOLUCION:
    print("Bicicleta:", solver.Value(bicicleta), "Casco:", solver.Value(casco))

Estado: OPTIMAL
Bicicleta: 1 Casco: 1


Ahora el casco cuesta 3 monedas. Si lo compro, debo tener al menos 3.

```python
model.Add(monedas >= 3).OnlyEnforceIf(compro_casco)
```

Se lee: **si `compro_casco = 1`, exige `monedas >= 3`**.
Si `compro_casco = 0`, esta línea no exige nada sobre las monedas.
Las otras restricciones y el dominio siguen vigentes.

También se puede escribir la implicación booleana bicicleta → casco como
`model.AddBoolOr([bicicleta.Not(), casco])`: prohíbe la combinación 1, 0.

Si una regla debe activarse cuando **dos** condiciones sean ciertas:
`model.Add(x == 3).OnlyEnforceIf([a, b])` se activa cuando **a Y b** valen 1.
Para activarla cuando **a O b** sea cierto, se puede agregar la misma consecuencia
con `.OnlyEnforceIf(a)` y con `.OnlyEnforceIf(b)`.

## 9. Reificación: una lucecita que cuenta la verdad de otra condición

Imagina una luz que debe encenderse **exactamente cuando el Proyecto A está en
el día 3**. Llamamos `b1` a esa luz. Esto es **reificar**: representar una condición
mediante una booleana.

```text
PA = 3                 b1 = 1
PA = 1, 2, 4 o 5       b1 = 0
```

El nombre `"A_es_3"` es solo una etiqueta. OR-Tools no entiende su significado
por leer ese nombre: debemos escribir el vínculo.

Primero anotamos:

```python
model.Add(PA == 3).OnlyEnforceIf(b1)
```

Dice: **si la luz está encendida, A debe estar en el día 3**.
Si la luz está apagada, esa línea no hace ninguna exigencia. Por sí sola permite:

| PA | b1 | ¿La primera línea lo permite? |
|---|---|---|
| 3 | 1 | Sí |
| 3 | 0 | **Sí: falta obligar a que la luz se encienda** |
| 2 | 1 | No |
| 2 | 0 | Sí |

Para completar el vínculo agregamos:

```python
model.Add(PA != 3).OnlyEnforceIf(b1.Not())
```

Dice: **si la luz está apagada, A debe estar en un día distinto de 3**.
Ahora `PA = 3, b1 = 0` queda prohibido. Si A está en 3, la luz tiene que encenderse.

Las dos líneas juntas expresan **`b1 ↔ (PA == 3)`**, leído «b1 es verdadero si y
solo si PA vale 3». Se llama **reificación completa o bidireccional**.
La primera línea sola es reificación parcial.

**`b1.Not()` no elimina los días 1, 2, 4 y 5.** Los permite con `b1 = 0`.
Lo que elimina es que la luz diga «no» mientras `PA` realmente vale 3.

Otra forma de entender por qué la segunda línea expresa la vuelta:
«si PA es 3, b1 no puede ser 0» equivale a «si b1 es 0, PA no puede ser 3».
Esta forma de dar vuelta una implicación negando ambos lados se llama
**contraposición**.

### Ver el hueco con un ejemplo que el solver no puede ocultar

Queremos la regla «si PA = 3, entonces PB = 4».
Forzaremos `PA = 3`, `PB = 2` y `b1 = 0` para comprobar si el modelo detecta el
incumplimiento. Estas fijaciones son solo una demostración.

Sin la segunda línea, la luz puede quedar apagada y la consecuencia no se activa.
Con la segunda línea, esa combinación es imposible.

In [9]:
for incluir_vuelta in [False, True]:
    print("\n¿Incluimos la línea con b1.Not()?", incluir_vuelta)
    model = cp_model.CpModel()
    PA = model.NewIntVar(1, 5, "PA")
    PB = model.NewIntVar(1, 5, "PB")
    b1 = model.NewBoolVar("PA_es_3")

    model.Add(PA == 3).OnlyEnforceIf(b1)
    if incluir_vuelta:  # Es un dato Python del experimento.
        model.Add(PA != 3).OnlyEnforceIf(b1.Not())
    model.Add(PB == 4).OnlyEnforceIf(b1)

    # Combinación tramposa: la condición es cierta, pero la luz está apagada.
    model.Add(PA == 3)
    model.Add(PB == 2)
    model.Add(b1 == 0)

    solver, status = resolver(model)
    if status in HAY_SOLUCION:
        print("El modelo incompleto permite PA=3, PB=2, b1=0.")
    elif status == cp_model.INFEASIBLE:
        print("El modelo completo rechaza esa combinación.")


¿Incluimos la línea con b1.Not()? False
Estado: OPTIMAL
El modelo incompleto permite PA=3, PB=2, b1=0.

¿Incluimos la línea con b1.Not()? True
Estado: INFEASIBLE
El modelo completo rechaza esa combinación.


La equivalencia completa es necesaria cuando queremos que la luz **refleje
exactamente** la condición, como en tu pregunta 4 o al contar coincidencias.
No hace falta convertir toda implicación en equivalencia: «si llevo bicicleta,
llevo casco» sigue siendo una regla de una sola dirección.

Con este patrón, omitir la vuelta deja una forma de saltarse la regla.
Hay otras maneras de representar una implicación, por ejemplo una tabla de
parejas permitidas; aquí seguimos las booleanas explícitas que pide tu evaluación.

### Negar bien una condición

| Condición | Su negación exacta |
|---|---|
| `x == 3` | `x != 3` |
| `x != 3` | `x == 3` |
| `x < 3` | `x >= 3` |
| `x <= 3` | `x > 3` |
| `x >= 3` | `x < 3` |

Negar «menor que» incluye el caso de igualdad. No lo dejes fuera.

## 10. Tu pregunta 4 completa: días de dos proyectos

`PA` y `PB` pueden valer 1, 2, 3, 4 o 5. Las reglas son:

| Antecedente: «si…» | Booleano que lo representa | Consecuencia: «entonces…» |
|---|---|---|
| `PA == 3` | `b1 ↔ (PA == 3)` | `PB == 4` |
| `PA == 4` | `b2 ↔ (PA == 4)` | `PB == 5` |
| `PB == 1` | `b3 ↔ (PB == 1)` | `PA == 5` |

Hay tres indicadores porque elegimos representar esos tres antecedentes.
Primero conectamos las luces con las condiciones; luego usamos cada luz para
activar su consecuencia. Al final pedimos resolver e imprimimos los valores.

In [10]:
model = cp_model.CpModel()
PA = model.NewIntVar(1, 5, "PA")
PB = model.NewIntVar(1, 5, "PB")
b1 = model.NewBoolVar("PA_es_3")
b2 = model.NewBoolVar("PA_es_4")
b3 = model.NewBoolVar("PB_es_1")

# Luces que reflejan exactamente sus condiciones.
model.Add(PA == 3).OnlyEnforceIf(b1)
model.Add(PA != 3).OnlyEnforceIf(b1.Not())

model.Add(PA == 4).OnlyEnforceIf(b2)
model.Add(PA != 4).OnlyEnforceIf(b2.Not())

model.Add(PB == 1).OnlyEnforceIf(b3)
model.Add(PB != 1).OnlyEnforceIf(b3.Not())

# Consecuencias de las tres reglas.
model.Add(PB == 4).OnlyEnforceIf(b1)
model.Add(PB == 5).OnlyEnforceIf(b2)
model.Add(PA == 5).OnlyEnforceIf(b3)

solver = cp_model.CpSolver()
status = solver.Solve(model)
print("Estado:", solver.StatusName(status))
if status in HAY_SOLUCION:
    print("PA:", solver.Value(PA), "PB:", solver.Value(PB))
    print("b1:", solver.Value(b1))
    print("b2:", solver.Value(b2))
    print("b3:", solver.Value(b3))

# Guardamos este modelo para ver TODAS sus soluciones en la parada 26.
modelo_proyectos = model
variables_proyectos = [PA, PB, b1, b2, b3]

Estado: OPTIMAL
PA: 1 PB: 2
b1: 0
b2: 0
b3: 0


**No agregues `AddExactlyOne([b1, b2, b3])` por tener tres luces.**
Ninguna regla dice que deba cumplirse exactamente uno de esos antecedentes.
`PA = 2, PB = 2` es válido y las tres luces valen 0.

| PA | PB | ¿Válido? | Por qué |
|---|---|---|---|
| 3 | 4 | Sí | Cumple la primera regla |
| 3 | 2 | No | La primera exige PB = 4 |
| 4 | 5 | Sí | Cumple la segunda |
| 5 | 1 | Sí | Cumple la tercera |
| 1 | 1 | **No** | PB = 1 exige PA = 5 |
| 1 | 4 | Sí | Ningún antecedente se cumple |

La pareja **PA = 1, PB = 1 no es una solución**: incumple la tercera regla.
PA = 1, PB = 2 sí es válida.

Tampoco agregamos `PB != 4` cuando `b1` es falso: la primera regla no dice
«PB solo puede ser 4 si PA es 3». Por eso `PA = 1, PB = 4` sigue permitido.

En el bloque final «de cuaderno» del solucionario modificado, las booleanas se
crean pero faltan sus vínculos con A y B. Copiar solo ese bloque deja las reglas
incompletas. La celda anterior sí incluye los vínculos.

## 11. ¿Por qué no escribir un `if` normal de Python?

Al construir el modelo todavía no conocemos PA. Por eso esto **no es el patrón
correcto** para una variable por resolver:

```python
# No ejecutar:
if PA == 3:
    model.Add(PB == 4)
```

Python intentaría decidir el `if` antes de que el solver elija los días.
Tampoco puedes poner `OnlyEnforceIf(PA == 3)`: esa comparación no es una booleana
del modelo. Usa el indicador y su vínculo.

Un `if` normal sí sirve para cosas ya conocidas:

```python
if duracion == 2:                 # duracion es un dato Python.
    print("La tarea dura dos días")

if status in HAY_SOLUCION:        # El solver ya terminó.
    print(solver.Value(PA))
```

Dentro del modelo escribes reglas sobre decisiones futuras. Después de resolver,
puedes usar sus valores numéricos en Python.

## 12. AllDifferent: repartir turnos sin repetir

`AddAllDifferent([a, b, c])` exige que los tres valores sean diferentes entre sí.
Es como poner un juguete distinto en cada cajita.

Volvamos a los tres médicos de tu PC2. Codificamos:
**1 = mañana, 2 = tarde, 3 = noche**.

- M1 no trabaja de noche: `m1 != 3`.
- M2 no trabaja de mañana: `m2 != 1`.
- M3 va después de M1 y antes de M2: `m1 < m3` y `m3 < m2`.
- No repiten turno: `AddAllDifferent`.

Aquí, al asignar tres valores distintos elegidos entre tres turnos, necesariamente
se usan los tres. Con dos médicos y tres turnos, `AllDifferent` no obligaría a
cubrirlos todos.

In [11]:
model = cp_model.CpModel()
m1 = model.NewIntVar(1, 3, "M1")
m2 = model.NewIntVar(1, 3, "M2")
m3 = model.NewIntVar(1, 3, "M3")

model.AddAllDifferent([m1, m2, m3])
model.Add(m1 != 3)
model.Add(m2 != 1)
model.Add(m1 < m3)
model.Add(m3 < m2)

solver, status = resolver(model)
if status in HAY_SOLUCION:
    turnos = {1: "mañana", 2: "tarde", 3: "noche"}
    print("M1:", turnos[solver.Value(m1)])
    print("M2:", turnos[solver.Value(m2)])
    print("M3:", turnos[solver.Value(m3)])
    print("Tiempo:", solver.WallTime(), "segundos")

Estado: OPTIMAL
M1: mañana
M2: noche
M3: tarde
Tiempo: 0.000513 segundos


**Resultado:** M1 mañana, M3 tarde y M2 noche.
Es único: el orden estricto `m1 < m3 < m2`, con solo los valores 1, 2 y 3,
fuerza 1 < 2 < 3.

En el vocabulario formal, una regla con una variable es **unaria** (`m1 != 3`),
con dos es **binaria** (`m1 < m3`). Una regla que expresa un patrón sobre un
conjunto, como `AllDifferent` o `Cumulative`, se llama **global**.

## 13. Listas de listas: una tabla de lucecitas

Ahora hay tres niños y tres juguetes. `x[i][j]` responde:
**«¿El niño i recibe el juguete j?»**.

| | Oso j=0 | Tren j=1 | Pelota j=2 |
|---|---|---|---|
| Ana i=0 | `x[0][0]` | `x[0][1]` | `x[0][2]` |
| Beto i=1 | `x[1][0]` | `x[1][1]` | `x[1][2]` |
| Cami i=2 | `x[2][0]` | `x[2][1]` | `x[2][2]` |

Cada **fila** recorre juguetes de un niño. Cada **columna** recorre niños para un
juguete. `x[0][1] = 1` significa «Ana recibe el tren».

Cada niño recibe **exactamente uno**: EO por fila.
Cada juguete se entrega **a lo sumo una vez**: AMO por columna.
Como hay tres niños y tres juguetes, todos se acabarán usando.

In [12]:
model = cp_model.CpModel()
ninos = ["Ana", "Beto", "Cami"]
juguetes = ["oso", "tren", "pelota"]
x = []

# Construir la tabla. Una fila por niño, una luz por juguete.
for i in range(len(ninos)):
    fila = []
    for j in range(len(juguetes)):
        fila.append(model.NewBoolVar(f"nino_{i}_juguete_{j}"))
    x.append(fila)

for i in range(len(ninos)):
    model.AddExactlyOne(x[i])

for j in range(len(juguetes)):
    columna = [x[i][j] for i in range(len(ninos))]
    model.AddAtMostOne(columna)

model.Add(x[0][1] == 0)  # Ana no puede recibir el tren.

solver, status = resolver(model)
if status in HAY_SOLUCION:
    for i in range(len(ninos)):
        for j in range(len(juguetes)):
            if solver.Value(x[i][j]) == 1:
                print(ninos[i], "recibe", juguetes[j])

Estado: OPTIMAL
Ana recibe oso
Beto recibe pelota
Cami recibe tren


La expresión `[x[i][j] for i in range(3)]` crea una lista con las tres posiciones
de la columna j. Es una forma corta de hacer una lista usando un `for`.
`f"nino_{i}"` inserta el número i en un texto para nombrar variables.

**Dos formas de representar una elección:** un entero `juguete_de_ana` entre
1 y 3 ya elige un único valor; tres booleanas necesitan EO para expresar esa
misma elección. Usa la representación que haga más fáciles tus reglas.

## 14. Horarios: cada cosa una vez y sin choques

La escuela de invierno de tus notebooks usa `sesion[curso][dia][turno][salon]`.
Aunque tiene cuatro índices, cada entrada sigue siendo una pregunta de sí/no.
`sesion[2][0][1][0]` pregunta por el curso 2, primer día, segundo turno, primer salón.

Practicaremos con dos actividades, dos turnos y un salón.
Cada actividad ocurre una vez (**EO por actividad**); en cada turno cabe a lo
sumo una actividad (**AMO por turno**).

Para una precedencia, «A antes de B», asignaremos a cada turno su posición temporal.
Si hay varios días y dos turnos por día, `posicion = dia * 2 + turno` permite
comparar la secuencia completa: 0, 1, 2 y 3.

In [13]:
model = cp_model.CpModel()
actividades = ["pintar", "cantar"]
horarios = ["mañana", "tarde"]
x = [[model.NewBoolVar(f"actividad_{i}_turno_{t}")
      for t in range(2)] for i in range(2)]

for i in range(2):
    model.AddExactlyOne(x[i])
for t in range(2):
    model.AddAtMostOne([x[0][t], x[1][t]])

# EO garantiza un solo 1: la suma recupera el turno seleccionado.
turno_pintar = sum(t * x[0][t] for t in range(2))
turno_cantar = sum(t * x[1][t] for t in range(2))
model.Add(turno_pintar < turno_cantar)

solver, status = resolver(model)
if status in HAY_SOLUCION:
    for i in range(2):
        for t in range(2):
            if solver.Value(x[i][t]):
                print(actividades[i], "->", horarios[t])

Estado: OPTIMAL
pintar -> mañana
cantar -> tarde


Para convertir una fila EO en un número: `[0, 0, 1]` elige la posición 2 porque
`0*0 + 1*0 + 2*1 = 2`. Si los días se numeran 1, 2 y 3, usa esos números como
pesos. Sin EO, esa suma podría combinar varios turnos y dejaría de representar
una elección única.

## 15. Grupos de equipos: repetir las mismas reglas a otra escala

Tus sorteos usan `g[equipo][grupo]`. Es la misma tabla de los juguetes.
Usaremos cuatro equipos de juguete, dos grupos y dos tipos de camiseta.

- Cada equipo pertenece a exactamente un grupo: **EO por fila**.
- Cada grupo tiene exactamente dos equipos: **suma de columna = 2**.
- Cada grupo tiene uno de cada tipo de camiseta: **suma filtrada = 1**.

La camiseta es un **dato conocido**, así que usamos una lista Python.
La pertenencia al grupo es una **decisión**, así que usamos booleanas.

In [14]:
model = cp_model.CpModel()
equipos = ["Osos", "Gatos", "Patos", "Ranas"]
camisetas = ["roja", "roja", "azul", "azul"]
g = [[model.NewBoolVar(f"equipo_{i}_grupo_{j}")
      for j in range(2)] for i in range(len(equipos))]

for i in range(len(equipos)):
    model.AddExactlyOne(g[i])

for j in range(2):
    model.Add(sum(g[i][j] for i in range(len(equipos))) == 2)
    for camiseta in ["roja", "azul"]:
        model.Add(sum(g[i][j] for i in range(len(equipos))
                      if camisetas[i] == camiseta) == 1)

solver, status = resolver(model)
if status in HAY_SOLUCION:
    for j in range(2):
        print("Grupo", j + 1)
        for i in range(len(equipos)):
            if solver.Value(g[i][j]):
                print(" ", equipos[i], "-", camisetas[i])

Estado: OPTIMAL
Grupo 1
  Gatos - roja
  Ranas - azul
Grupo 2
  Osos - roja
  Patos - azul


El `if camisetas[i] == camiseta` sí se puede usar: compara textos ya conocidos.
En los ejercicios grandes, el tipo de camiseta se parece al **bombo** o a la
**federación**. Para un límite de dos miembros, cambia `== 1` por `<= 2`.

Al cambiar la cantidad de equipos, usa `len(equipos)` y revisa también grupos,
tamaños y datos. Un `range(32)` dejado en un caso de 48 equipos ignora los últimos
16: ese problema aparece en los conteos del ejercicio grande del repositorio.
Aquí usamos equipos inventados para concentrarnos en las reglas.

## 16. Contar cuántas variables valen algo

¿Cuántas cajitas contienen el número 3? Si son decisiones todavía desconocidas,
creamos una luz por cajita: **1 exactamente cuando esa cajita vale 3**.
Después sumamos las luces.

```text
Cajitas:  [3, 1, 3, 2]
Luces:    [1, 0, 1, 0]
Suma:     2
```

Aquí aparece de nuevo la reificación completa: sin la vuelta, una cajita con 3
podría tener luz 0 y no contaríamos bien. La función devuelve un contador del
modelo, que después podemos usar en otras reglas.

In [15]:
def contar_iguales(modelo, variables, valor, nombre):
    luces = []
    for i in range(len(variables)):
        b = modelo.NewBoolVar(f"{nombre}_luz_{i}")
        modelo.Add(variables[i] == valor).OnlyEnforceIf(b)
        modelo.Add(variables[i] != valor).OnlyEnforceIf(b.Not())
        luces.append(b)
    contador = modelo.NewIntVar(0, len(variables), nombre)
    modelo.Add(contador == sum(luces))
    return contador


model = cp_model.CpModel()
cajitas = [model.NewIntVar(1, 3, f"cajita_{i}") for i in range(4)]
cantidad_de_tres = contar_iguales(model, cajitas, 3, "cantidad_de_tres")
model.Add(cantidad_de_tres == 2)

solver, status = resolver(model)
if status in HAY_SOLUCION:
    print("Cajitas:", [solver.Value(v) for v in cajitas])
    print("Cantidad de tres:", solver.Value(cantidad_de_tres))

Estado: OPTIMAL
Cajitas: [1, 1, 3, 3]
Cantidad de tres: 2


La salida puede variar, pero **exactamente dos cajitas deben tener 3**.
`return` entrega el resultado de una función. En tus notebooks esta herramienta
se llama `count_vars`.

## 17. Multiplicar por una luz: incluir o quitar una cantidad

Una caja pesa 3 kilos. Si la llevamos, cuenta 3; si no, cuenta 0.

```text
3 × 1 = 3       3 × 0 = 0
```

Multiplicar un **dato conocido** por una booleana se puede escribir directamente:
`model.Add(peso == 3 * llevar)`.

Si ambos factores son **variables del modelo**, usamos
`AddMultiplicationEquality(resultado, [factor1, factor2])`.
Así lo hacen tus notebooks al multiplicar federación por pertenencia a un grupo.
El código de una federación es una etiqueta; no expresa que sea mejor o peor.

In [16]:
for llevar_caja in [0, 1]:
    model = cp_model.CpModel()
    peso_caja = model.NewIntVar(1, 5, "peso_caja")
    llevar = model.NewBoolVar("llevar")
    peso_transportado = model.NewIntVar(0, 5, "peso_transportado")

    model.Add(peso_caja == 3)
    model.Add(llevar == llevar_caja)
    model.AddMultiplicationEquality(peso_transportado, [peso_caja, llevar])

    solver, status = resolver(model)
    if status in HAY_SOLUCION:
        print("¿Llevar?", llevar_caja, "Peso transportado:", solver.Value(peso_transportado))

Estado: OPTIMAL
¿Llevar? 0 Peso transportado: 0
Estado: OPTIMAL
¿Llevar? 1 Peso transportado: 3


## 18. Tablas: una lista de combinaciones permitidas o prohibidas

Para la merienda acordamos:

| Comida | Bebida |
|---|---|
| 1 = pan | 1 = leche |
| 2 = fruta | 2 = agua |

Solo esas parejas están permitidas. `AddAllowedAssignments` escribe esa lista.
`AddForbiddenAssignments` escribe las combinaciones que deben quedar fuera.

El orden importa: `(comida, bebida)` no es `(bebida, comida)`.
Cada fila debe tener tantos valores como variables tiene la lista.
Una tabla permitida vacía no permite ninguna combinación.

In [17]:
model = cp_model.CpModel()
comida = model.NewIntVar(1, 2, "comida")
bebida = model.NewIntVar(1, 2, "bebida")

model.AddAllowedAssignments([comida, bebida], [(1, 1), (2, 2)])
model.AddForbiddenAssignments([comida, bebida], [(1, 1)])

solver, status = resolver(model)
if status in HAY_SOLUCION:
    print("Comida:", solver.Value(comida), "Bebida:", solver.Value(bebida))
    print("Quedó fruta con agua: (2, 2).")

Estado: OPTIMAL
Comida: 2 Bebida: 2
Quedó fruta con agua: (2, 2).


Estas son **restricciones de tabla** o **extensionales**: enumeran los casos.
En cambio, `x < y` describe una regla mediante una expresión.
En tu notebook de texto, las tablas dicen qué palabra puede seguir a otra y qué
finales pueden rimar.

## 19. Grafos y colores: vecinos con camisetas distintas

Un **grafo** es un dibujo de puntos unidos por líneas. Cada punto es un nodo;
cada línea indica que dos nodos son vecinos.

```text
A ---- B ---- C
```

A y B deben tener colores distintos. B y C también. A y C pueden compartir color
porque no son vecinos. No queremos `AllDifferent` para todos: eso añadiría una
regla que el dibujo no pide.

Esta es la misma idea de pintar regiones del mapa de Perú sin que dos vecinas
tengan el mismo color.

In [18]:
model = cp_model.CpModel()
color = [model.NewIntVar(1, 2, f"color_{i}") for i in range(3)]
aristas = [(0, 1), (1, 2)]  # A-B y B-C.
for u, v in aristas:
    model.Add(color[u] != color[v])

solver, status = resolver(model)
if status in HAY_SOLUCION:
    nombres_colores = {1: "rojo", 2: "azul"}
    for i in range(3):
        print(["A", "B", "C"][i], "->", nombres_colores[solver.Value(color[i])])

Estado: OPTIMAL
A -> rojo
B -> azul
C -> rojo


### ¿Y si el grafo es lo que tenemos que descubrir?

Otro ejercicio de tus notebooks crea una tabla booleana de vecinos.
`vecino[i][j] = 1` significa que i está conectado con j.
Para un grafo sin flechas, `vecino[i][j] == vecino[j][i]` hace que la relación
funcione en ambos sentidos. `vecino[i][i] == 0` evita conectarse consigo mismo.

Con los números de nodo 1, 2 y 3, pedimos que cada nodo tenga estas sumas de
vecinos: `[2, 4, 2]`. Una suma ponderada cuenta el **número del vecino** cuando
su luz está encendida.

In [19]:
model = cp_model.CpModel()
sumas_vecinos = [2, 4, 2]
vecino = [[model.NewBoolVar(f"vecino_{i}_{j}") for j in range(3)]
          for i in range(3)]

for i in range(3):
    model.Add(vecino[i][i] == 0)
    model.Add(sum((j + 1) * vecino[i][j] for j in range(3)) == sumas_vecinos[i])
    for j in range(3):
        model.Add(vecino[i][j] == vecino[j][i])

solver, status = resolver(model)
if status in HAY_SOLUCION:
    for i in range(3):
        sus_vecinos = [j + 1 for j in range(3) if solver.Value(vecino[i][j])]
        print("Nodo", i + 1, "Vecinos:", sus_vecinos, "Suma:", sum(sus_vecinos))

Estado: OPTIMAL
Nodo 1 Vecinos: [2] Suma: 2
Nodo 2 Vecinos: [1, 3] Suma: 4
Nodo 3 Vecinos: [2] Suma: 2


## 20. Cuadrículas y letras: las mismas herramientas en otros juegos

### Cuadrado latino: no repetir en filas ni columnas

Pon los números 1, 2 y 3 en una cuadrícula de 3 × 3. Cada fila y cada columna
debe tenerlos sin repetir. Aplicamos `AllDifferent` a cada fila y columna.
No hay reglas para subcuadros como en un sudoku.

In [20]:
model = cp_model.CpModel()
n = 3
cuadricula = [[model.NewIntVar(1, n, f"celda_{i}_{j}") for j in range(n)]
              for i in range(n)]
for i in range(n):
    model.AddAllDifferent(cuadricula[i])
for j in range(n):
    model.AddAllDifferent([cuadricula[i][j] for i in range(n)])

solver, status = resolver(model)
if status in HAY_SOLUCION:
    for fila in cuadricula:
        print([solver.Value(v) for v in fila])

Estado: OPTIMAL
[1, 3, 2]
[2, 1, 3]
[3, 2, 1]


### Cuadrado mágico: además, sumas iguales

Ahora usamos del 1 al 9, cada número una vez. En un cuadrado mágico de 3 × 3,
cada fila, cada columna **y las dos diagonales** suman 15.

En el ejemplo del curso se restringen filas y columnas. Aquí agregamos también
las diagonales para cumplir la definición habitual de cuadrado mágico.

In [21]:
model = cp_model.CpModel()
cuadrado = [[model.NewIntVar(1, 9, f"magico_{i}_{j}") for j in range(3)]
            for i in range(3)]
todas = [v for fila in cuadrado for v in fila]  # Aplanar la tabla.
model.AddAllDifferent(todas)
for i in range(3):
    model.Add(sum(cuadrado[i]) == 15)
for j in range(3):
    model.Add(sum(cuadrado[i][j] for i in range(3)) == 15)
model.Add(sum(cuadrado[i][i] for i in range(3)) == 15)
model.Add(sum(cuadrado[i][2 - i] for i in range(3)) == 15)

solver, status = resolver(model)
if status in HAY_SOLUCION:
    for fila in cuadrado:
        print([solver.Value(v) for v in fila])

Estado: OPTIMAL
[8, 1, 6]
[3, 5, 7]
[4, 9, 2]


### Criptograma: una letra es una cajita con un dígito

En `SEND + MORE = MONEY`, cada letra representa un dígito distinto.
La S y la M no pueden ser 0 porque empiezan números.
El 1000 significa «esta letra ocupa el lugar de los miles»:
`SEND = 1000*S + 100*E + 10*N + D`.

Es un ejemplo más grande, pero la receta sigue siendo crear cajitas, anotar
reglas y resolver.

In [22]:
model = cp_model.CpModel()
letras = {}
for letra in "SENDMORY":
    minimo = 1 if letra in "SM" else 0
    letras[letra] = model.NewIntVar(minimo, 9, letra)

S, E, N, D, M, O, R, Y = [letras[l] for l in "SENDMORY"]
model.AddAllDifferent(list(letras.values()))
send = 1000*S + 100*E + 10*N + D
more = 1000*M + 100*O + 10*R + E
money = 10000*M + 1000*O + 100*N + 10*E + Y
model.Add(send + more == money)

solver, status = resolver(model)
if status in HAY_SOLUCION:
    print(solver.Value(send), "+", solver.Value(more), "=", solver.Value(money))

Estado: OPTIMAL
9567 + 1085 = 10652


### Anagramas: elegir una palabra con las mismas letras

Un anagrama cambia el orden, pero conserva las letras y sus cantidades.
«amor» y «roma» tienen una a, una m, una o y una r.
Creamos una luz por palabra candidata y exigimos EO para elegir una palabra.
Después comprobamos el conteo de cada letra mediante sumas ponderadas.

In [23]:
from collections import Counter

objetivo = "amor"
diccionario = ["roma", "ramo", "mora", "casa", "perro"]
candidatas = [p for p in diccionario if len(p) == len(objetivo)]
alfabeto = sorted(set(objetivo + "".join(candidatas)))
conteo = Counter(objetivo)

model = cp_model.CpModel()
elegida = [model.NewBoolVar(f"palabra_{i}") for i in range(len(candidatas))]
model.AddExactlyOne(elegida)
for letra in alfabeto:
    model.Add(sum(candidatas[i].count(letra) * elegida[i]
                  for i in range(len(candidatas))) == conteo[letra])

solver, status = resolver(model)
if status in HAY_SOLUCION:
    for i in range(len(candidatas)):
        if solver.Value(elegida[i]):
            print("Un anagrama de", objetivo, "es", candidatas[i])

Estado: OPTIMAL
Un anagrama de amor es mora


## 21. Palabras como números: una secuencia con reglas

El notebook de texto usa un corpus: un texto del que saca palabras y parejas
consecutivas. Para entender la parte CSP, aquí tenemos todos los datos a la vista.

| ID | Palabra |
|---|---|
| 0 | mira |
| 1 | gato |
| 2 | salta |
| 3 | pato |

Queremos cuatro posiciones. Las tablas indican qué palabra puede seguir a otra.
Los finales de las dos líneas deben ser «gato» y «pato», en cualquier orden.
Además, no repetimos palabras y prohibimos una secuencia concreta.
Es una aproximación muy pequeña al ejercicio de rimas, no un sistema que entienda
poesía.

In [24]:
palabras = ["mira", "gato", "salta", "pato"]
# mira->gato, gato->salta, salta->pato,
# mira->pato, pato->salta, salta->gato.
transiciones = [(0, 1), (1, 2), (2, 3), (0, 3), (3, 2), (2, 1)]
finales_que_riman = [(1, 3), (3, 1)]

model = cp_model.CpModel()
secuencia = [model.NewIntVar(0, 3, f"palabra_en_posicion_{p}") for p in range(4)]
model.Add(secuencia[0] == 0)  # Empezar con "mira".
for p in range(3):
    model.AddAllowedAssignments([secuencia[p], secuencia[p + 1]], transiciones)
model.AddAllowedAssignments([secuencia[1], secuencia[3]], finales_que_riman)
model.AddAllDifferent(secuencia)
model.AddForbiddenAssignments(secuencia, [(0, 1, 2, 3)])

solver, status = resolver(model)
if status in HAY_SOLUCION:
    ids = [solver.Value(v) for v in secuencia]
    print("IDs:", ids)
    print("Línea 1:", palabras[ids[0]], palabras[ids[1]])
    print("Línea 2:", palabras[ids[2]], palabras[ids[3]])

Estado: OPTIMAL
IDs: [0, 3, 2, 1]
Línea 1: mira pato
Línea 2: salta gato


**Resultado:** «mira pato / salta gato». La otra ruta, «mira gato / salta pato»,
está en la tabla de secuencias prohibidas.

En el ejemplo grande, cada posición también guarda un ID. Hay tablas para
transiciones, rimas y ventanas de palabras prohibidas. Cumplir esas reglas no
garantiza que el texto tenga sentido. Prohibir algunas secuencias tampoco
garantiza que no haya plagio.

El notebook de GNU Plot enseña a dibujar funciones y datos, no a construir
un CSP. Para este repaso bastan las tablas y los dibujos pequeños incluidos.

## 22. Tareas que duran tiempo: inicio, duración y fin

Una tarea es como jugar con plastilina durante dos horas.
Si empiezas a las 3 y dura 2, terminas a las 5:

```text
inicio + duración = fin
   3   +     2    =  5

Hora:       3       4       5
            [ plastilina   )
```

Un **intervalo** junta inicio, duración y fin. `NewIntervalVar` impone esa
igualdad. Se ocupa el tiempo **desde el inicio hasta antes del fin**: `[inicio, fin)`.
Una tarea que termina en 5 permite que otra empiece en 5 sin superponerse.

### Una sola máquina: NoOverlap

Si tenemos un solo horno, dos tortas no pueden hornearse a la vez.
`AddNoOverlap` exige que los intervalos no se superpongan.

In [25]:
model = cp_model.CpModel()
inicio_a = model.NewIntVar(0, 4, "inicio_torta_a")
fin_a = model.NewIntVar(2, 6, "fin_torta_a")
horno_a = model.NewIntervalVar(inicio_a, 2, fin_a, "horno_a")
inicio_b = model.NewIntVar(0, 3, "inicio_torta_b")
fin_b = model.NewIntVar(3, 6, "fin_torta_b")
horno_b = model.NewIntervalVar(inicio_b, 3, fin_b, "horno_b")
model.AddNoOverlap([horno_a, horno_b])

solver, status = resolver(model)
if status in HAY_SOLUCION:
    print("Torta A:", solver.Value(inicio_a), "hasta", solver.Value(fin_a))
    print("Torta B:", solver.Value(inicio_b), "hasta", solver.Value(fin_b))

Estado: OPTIMAL
Torta A: 3 hasta 5
Torta B: 0 hasta 3


## 23. Cumulative: varios pueden trabajar juntos si alcanza el recurso

En tu obra hay **5 obreros** disponibles. Las tareas son:

| Tarea | Duración | Obreros que ocupa mientras trabaja |
|---|---|---|
| 1 | 2 días | 3 |
| 2 | 3 días | 2 |
| 3 | 1 día | 4 |

La tarea 1 y la 2 sí pueden coincidir: `3 + 2 = 5`.
La 1 y la 3 no: `3 + 4 = 7`, que supera 5.

**`AddCumulative` exige que en todo momento la suma de demandas de las tareas
activas no supere la capacidad.** Piensa en mirar el calendario y contar cuántos
obreros se necesitan juntos. Esto describe la regla; el solver emplea algoritmos
de propagación y búsqueda para cumplirla.

`NoOverlap` prohibiría también la coincidencia permitida de las tareas 1 y 2.
Y comprobar únicamente que cada pareja cabe no sirve en general: tres tareas
de demanda 2 caben de dos en dos (`2 + 2 <= 5`), pero las tres juntas no (`6 > 5`).
Necesitamos controlar el total.

El horizonte 10 significa terminar como máximo en el instante 10. Los bloques
de día son 0–1, 1–2, …, 9–10: **diez bloques**.

In [26]:
model = cp_model.CpModel()
horizonte = 10
duraciones = [2, 3, 1]
demandas = [3, 2, 4]
capacidad = 5
inicios, fines, intervalos = [], [], []

for i in range(3):
    # Si dura 2, el último inicio posible es 8 para terminar en 10.
    inicio = model.NewIntVar(0, horizonte - duraciones[i], f"inicio_{i}")
    fin = model.NewIntVar(duraciones[i], horizonte, f"fin_{i}")
    intervalo = model.NewIntervalVar(inicio, duraciones[i], fin, f"tarea_{i}")
    inicios.append(inicio)
    fines.append(fin)
    intervalos.append(intervalo)

model.AddCumulative(intervalos, demandas, capacidad)

solver, status = resolver(model)
if status in HAY_SOLUCION:
    inicio_real = [solver.Value(s) for s in inicios]
    fin_real = [solver.Value(e) for e in fines]
    for i in range(3):
        print("Tarea", i + 1, "Inicio:", inicio_real[i], "Fin:", fin_real[i])
    print("\nEl calendario. # significa que esa tarea trabaja en ese día.")
    print("Día:   ", " ".join(str(t) for t in range(horizonte)))
    for i in range(3):
        fila = ["#" if inicio_real[i] <= t < fin_real[i] else "."
                for t in range(horizonte)]
        print("Tarea", i + 1, " ".join(fila))
    uso = [sum(demandas[i] for i in range(3)
               if inicio_real[i] <= t < fin_real[i]) for t in range(horizonte)]
    print("Uso:   ", " ".join(str(v) for v in uso))
    print("Capacidad:", capacidad)

Estado: OPTIMAL
Tarea 1 Inicio: 0 Fin: 2
Tarea 2 Inicio: 0 Fin: 3
Tarea 3 Inicio: 3 Fin: 4

El calendario. # significa que esa tarea trabaja en ese día.
Día:    0 1 2 3 4 5 6 7 8 9
Tarea 1 # # . . . . . . . .
Tarea 2 # # # . . . . . . .
Tarea 3 . . . # . . . . . .
Uso:    5 5 2 4 0 0 0 0 0 0
Capacidad: 5


La fila «Uso» debe contener solo números de 0 a 5. Puede haber varios calendarios
correctos. No se pidió terminar lo antes posible, solo terminar dentro del plazo.

## 24. La misma obra con luces para cada posible inicio

El solucionario modificado también muestra una forma manual para tiempos enteros.
`empieza[i, t]` dice «la tarea i empieza exactamente en t».

1. Creamos solo los inicios que permiten terminar dentro del horizonte.
2. Exigimos **EO** entre los inicios de cada tarea.
3. Para cada día, sumamos los obreros de las tareas que estarían activas.

Si una tarea empieza en `s` y dura `d`, está activa en el día `t` cuando
**`s <= t < s + d`**. Por ejemplo, inicio 2 y duración 3 ocupa los días 2, 3 y 4;
ya no ocupa el día 5.

La expresión `demanda * empieza[i, s]` cuenta la demanda solo si se eligió ese
inicio. No usamos `if` sobre la booleana: el `if` de abajo compara índices y
duraciones ya conocidos para elegir qué términos incluir en la suma.

In [27]:
model = cp_model.CpModel()
horizonte = 10
duraciones = [2, 3, 1]
demandas = [3, 2, 4]
capacidad = 5
empieza = {}  # Diccionario: su clave es la pareja (tarea, inicio).
inicios_posibles = []

for i in range(3):
    posibles = list(range(horizonte - duraciones[i] + 1))
    inicios_posibles.append(posibles)
    for s in posibles:
        empieza[i, s] = model.NewBoolVar(f"tarea_{i}_empieza_{s}")
    model.AddExactlyOne([empieza[i, s] for s in posibles])

for t in range(horizonte):
    obreros_del_dia = []
    for i in range(3):
        for s in inicios_posibles[i]:
            if s <= t < s + duraciones[i]:
                obreros_del_dia.append(demandas[i] * empieza[i, s])
    model.Add(sum(obreros_del_dia) <= capacidad)

solver, status = resolver(model)
if status in HAY_SOLUCION:
    for i in range(3):
        inicio = sum(s * solver.Value(empieza[i, s]) for s in inicios_posibles[i])
        print("Tarea", i + 1, "Inicio:", inicio, "Fin:", inicio + duraciones[i])

Estado: OPTIMAL
Tarea 1 Inicio: 8 Fin: 10
Tarea 2 Inicio: 7 Fin: 10
Tarea 3 Inicio: 0 Fin: 1


Las dos formulaciones controlan la misma capacidad en este ejemplo de días
enteros y duraciones constantes. `AddCumulative` expresa la regla de forma
compacta y aprovecha una restricción especializada. El modelo manual crea una
luz por cada posible inicio: si el horizonte crece, crece la tabla.

Para responder la parte del examen que pide **inicio, fin, intervalo y
AddCumulative**, usa la parada 23. La parada 24 ayuda a entender la otra versión.

## 25. Encontrar una solución y elegir la mejor son cosas diferentes

Hasta ahora pedíamos «una respuesta que cumpla las reglas». Eso es **satisfacción**.
Si además pedimos «la más divertida» o «la que cuesta menos», hacemos **optimización**.

Hay tres juguetes. Elegiremos exactamente dos con un presupuesto de 7 monedas.
Cada uno tiene un costo y unos puntos de diversión inventados.

| Juguete | Costo | Diversión |
|---|---|---|
| Oso | 3 | 6 |
| Tren | 4 | 9 |
| Pelota | 2 | 3 |

`Maximize` pide la mayor diversión posible. `Minimize` pediría el menor valor
de una expresión. El objetivo no da permiso para romper el presupuesto.

In [28]:
model = cp_model.CpModel()
juguetes = ["oso", "tren", "pelota"]
costos = [3, 4, 2]
diversion = [6, 9, 3]
elegidos = [model.NewBoolVar(f"elegido_{i}") for i in range(3)]

model.Add(sum(elegidos) == 2)
costo_total = sum(costos[i] * elegidos[i] for i in range(3))
puntos = sum(diversion[i] * elegidos[i] for i in range(3))
model.Add(costo_total <= 7)
model.Maximize(puntos)

solver, status = resolver(model)
if status in HAY_SOLUCION:
    print("Elegidos:", [juguetes[i] for i in range(3) if solver.Value(elegidos[i])])
    print("Costo:", solver.Value(costo_total))
    print("Diversión:", solver.Value(puntos))

Estado: OPTIMAL
Elegidos: ['oso', 'tren']
Costo: 7
Diversión: 15


**Resultado óptimo:** oso y tren, costo 7 y diversión 15.
Si el estado fuera solo `FEASIBLE`, sabríamos que hay una respuesta válida,
pero todavía no que sea la mejor.

En el mapa del curso se crean luces que dicen si se usa cada color y se minimiza
su suma. Para terminar una obra pronto, se puede crear `ultimo_fin`, relacionarlo
con los finales usando `model.AddMaxEquality(ultimo_fin, fines)` y después
escribir `model.Minimize(ultimo_fin)`.

CP-SAT usa variables enteras. Para precios con decimales, puedes usar céntimos:
1,50 soles se representa como 150 céntimos.

## 26. ¿Una solución o todas? El callback

Un **callback** es una acción que el solver llama cada vez que encuentra una
solución. Imagina una campanita: «¡encontré otra!».
La siguiente clase guarda cuántas encontró y muestra sus valores.

`class` crea un tipo de herramienta; `self` se refiere a esa herramienta;
`__init__` la prepara. No necesitas memorizar la clase: puedes reutilizarla.
Dentro del callback, `self.Value(v)` lee la solución de ese momento.

Para enumerar usamos un modelo **sin objetivo**, `enumerate_all_solutions = True`
y un solo trabajador. Ponemos límites para no imprimir miles de respuestas.
Si se corta la búsqueda, lo mostrado es una muestra, no todas las soluciones.

In [29]:
class MostrarSoluciones(cp_model.CpSolverSolutionCallback):
    def __init__(self, variables, limite=100):
        super().__init__()
        self.variables = variables
        self.limite = limite
        self.cantidad = 0
        self.corto_por_limite = False

    def on_solution_callback(self):
        self.cantidad += 1
        valores = [self.Value(v) for v in self.variables]
        print("Solución", self.cantidad, ":", valores)
        if self.cantidad >= self.limite:
            self.corto_por_limite = True
            self.StopSearch()


def enumerar(modelo, variables, limite=100):
    error = modelo.Validate()
    if error:
        raise ValueError(error)
    solver = cp_model.CpSolver()
    solver.parameters.enumerate_all_solutions = True
    solver.parameters.num_search_workers = 1
    solver.parameters.max_time_in_seconds = 10
    campanita = MostrarSoluciones(variables, limite)
    estado = solver.Solve(modelo, campanita)
    print("Estado:", solver.StatusName(estado))
    print("Encontradas:", campanita.cantidad)
    if campanita.corto_por_limite:
        print("Se alcanzó el límite: no afirmamos haber mostrado todas.")
    elif estado == cp_model.OPTIMAL:
        print("La enumeración terminó: se mostraron todas.")
    elif estado == cp_model.INFEASIBLE:
        print("Se demostró que no hay soluciones.")
    else:
        print("La búsqueda no se completó: podría haber más soluciones.")
    return campanita

### Ver todas las selecciones ALO, AMO y EO

Ejecuta y compara con la tabla de la parada 6. Con tres juguetes, esperamos
**7 soluciones ALO, 4 AMO y 3 EO**. Cada solución es una fila de tres ceros o unos.

In [30]:
for regla in ["ALO", "AMO", "EO"]:
    print("\n---", regla, "---")
    model = cp_model.CpModel()
    luces = [model.NewBoolVar(f"juguete_{i}") for i in range(3)]
    if regla == "ALO":
        model.AddBoolOr(luces)
    elif regla == "AMO":
        model.AddAtMostOne(luces)
    else:
        model.AddExactlyOne(luces)
    enumerar(model, luces)


--- ALO ---
Solución 1 : [0, 1, 0]
Solución 2 : [0, 1, 1]
Solución 3 : [0, 0, 1]
Solución 4 : [1, 0, 1]
Solución 5 : [1, 0, 0]
Solución 6 : [1, 1, 0]
Solución 7 : [1, 1, 1]
Estado: OPTIMAL
Encontradas: 7
La enumeración terminó: se mostraron todas.

--- AMO ---
Solución 1 : [0, 0, 0]
Solución 2 : [0, 1, 0]
Solución 3 : [0, 0, 1]
Solución 4 : [1, 0, 0]
Estado: OPTIMAL
Encontradas: 4
La enumeración terminó: se mostraron todas.

--- EO ---
Solución 1 : [0, 1, 0]
Solución 2 : [0, 0, 1]
Solución 3 : [1, 0, 0]
Estado: OPTIMAL
Encontradas: 3
La enumeración terminó: se mostraron todas.


### Ver todas las soluciones de la pregunta 4

Usamos el modelo guardado en la parada 10. Cada fila se imprime en el orden
**`[PA, PB, b1, b2, b3]`**.

In [31]:
print("Orden: PA, PB, b1, b2, b3")
campanita_proyectos = enumerar(modelo_proyectos, variables_proyectos)

Orden: PA, PB, b1, b2, b3
Solución 1 : [5, 2, 0, 0, 0]
Solución 2 : [5, 1, 0, 0, 1]
Solución 3 : [1, 2, 0, 0, 0]
Solución 4 : [1, 3, 0, 0, 0]
Solución 5 : [1, 4, 0, 0, 0]
Solución 6 : [1, 5, 0, 0, 0]
Solución 7 : [2, 5, 0, 0, 0]
Solución 8 : [2, 4, 0, 0, 0]
Solución 9 : [4, 5, 0, 1, 0]
Solución 10 : [5, 5, 0, 0, 0]
Solución 11 : [5, 4, 0, 0, 0]
Solución 12 : [5, 3, 0, 0, 0]
Solución 13 : [2, 3, 0, 0, 0]
Solución 14 : [2, 2, 0, 0, 0]
Solución 15 : [3, 4, 1, 0, 0]
Estado: OPTIMAL
Encontradas: 15
La enumeración terminó: se mostraron todas.


Hay **15 parejas válidas**. Podemos contarlas sin programa:

| PA | PB permitido | Cantidad |
|---|---|---|
| 1 | 2, 3, 4, 5 | 4 |
| 2 | 2, 3, 4, 5 | 4 |
| 3 | 4 | 1 |
| 4 | 5 | 1 |
| 5 | 1, 2, 3, 4, 5 | 5 |

Total: `4 + 4 + 1 + 1 + 5 = 15`. Para cada pareja, los tres indicadores tienen
valores determinados por sus equivalencias.

En tus apuntes también aparece `SearchForAllSolutions`; esta versión del repaso
usa `Solve(model, callback)` con el parámetro de enumeración.
No enumeres directamente un modelo con `Maximize` o `Minimize` usando esta receta.

## 27. Cuando las reglas chocan: aprender a leer el resultado

Si pedimos que una cajita valga 1 y a la vez 2, no hay forma de ganar el juego.
El solver devuelve `INFEASIBLE`; no es un error de Python.

In [32]:
model = cp_model.CpModel()
cajita = model.NewIntVar(1, 2, "cajita")
model.Add(cajita == 1)
model.Add(cajita == 2)
solver, status = resolver(model)
if status == cp_model.INFEASIBLE:
    print("Estas dos reglas son incompatibles.")

Estado: INFEASIBLE
Estas dos reglas son incompatibles.


El solver puede descartar opciones por las reglas (**propagación**) y explorar
alternativas (**búsqueda**). Si Ana elige oso y no pueden repetir, Beto ya no
puede elegir oso. No tienes que programar manualmente todas las combinaciones.

| Lo que ves | Qué revisar |
|---|---|
| Nadie recibe nada | Quizá pusiste AMO donde necesitabas EO o ALO |
| Se eligen varios juguetes | Quizá pusiste ALO donde necesitabas EO |
| PA es 3, pero la regla no se activa | Revisa el vínculo de b1 y su vuelta |
| Conteo menor que las coincidencias reales | Revisa la reificación completa de cada indicador |
| `INFEASIBLE` | Reglas que no pueden cumplirse juntas o dominios demasiado pequeños |
| `UNKNOWN` | La búsqueda no concluyó; no demuestra imposibilidad |
| Faltan equipos al pasar de 32 a 48 | Revisa los límites de todos los bucles |
| Índice fuera de rango | Una lista de tamaño n llega hasta n−1 |
| La respuesta cambia | Puede haber varias soluciones correctas |
| No aparece ninguna respuesta impresa | ¿Llamaste a `Solve` y comprobaste el estado? |

Construye el modelo por partes: primero dominios; después una familia de reglas;
luego otra. Si una parte nueva hace imposible el modelo, revisa cómo se combina
con las anteriores. `model.Validate()` revisa la estructura, no la existencia de
soluciones.

### Un pequeño diccionario para consultar

| Quiero decir… | Escribo… |
|---|---|
| Una decisión numérica entre a y b, incluidos | `model.NewIntVar(a, b, "nombre")` |
| Una decisión de sí o no | `model.NewBoolVar("nombre")` |
| Exigir una regla | `model.Add(regla)` |
| Al menos un sí | `model.AddBoolOr(lista)` |
| A lo sumo un sí | `model.AddAtMostOne(lista)` |
| Exactamente un sí | `model.AddExactlyOne(lista)` |
| Todos sí | `model.AddBoolAnd(lista)` |
| Negación de una booleana | `b.Not()` o `~b` |
| Si a, entonces b, ambas booleanas | `model.AddImplication(a, b)` |
| Si b, exigir una regla numérica | `model.Add(regla).OnlyEnforceIf(b)` |
| Valores distintos | `model.AddAllDifferent(lista)` |
| Permitir / prohibir combinaciones | `AddAllowedAssignments` / `AddForbiddenAssignments` |
| Producto de variables | `model.AddMultiplicationEquality(z, [x, y])` |
| Inicio + duración = fin | `model.NewIntervalVar(inicio, duracion, fin, "nombre")` |
| Ninguna tarea se superpone | `model.AddNoOverlap(intervalos)` |
| La demanda simultánea cabe | `model.AddCumulative(intervalos, demandas, capacidad)` |
| Mejor puntuación / menor costo | `model.Maximize(expr)` / `model.Minimize(expr)` |
| Buscar la respuesta | `solver.Solve(model)` |
| Leer una elección encontrada | `solver.Value(variable)` |
| Leer tiempo de búsqueda | `solver.WallTime()` |

Los nombres de operadores lógicos también se escriben como símbolos:
`¬` = NO, `∧` = Y, `∨` = O, `→` = implica, `↔` = si y solo si.

## 28. Pequeños retos con respuestas

Intenta decir la respuesta con tus palabras antes de abrirla.

### Reto 1: la mochila vacía

Si ponemos AMO para tres juguetes, ¿puede quedar la mochila vacía?
¿Qué regla pondrías para que tenga exactamente un juguete?

<details>
<summary>Ver respuesta</summary>

Sí. AMO permite 0 o 1. Para uno obligatorio: EO, `AddExactlyOne(juguetes)`
o `Add(sum(juguetes) == 1)`.

</details>

### Reto 2: varios juguetes

¿La selección `[1, 0, 1]` cumple ALO? ¿AMO? ¿EO?

<details>
<summary>Ver respuesta</summary>

Tiene dos juguetes: cumple ALO, pero incumple AMO y EO.

</details>

### Reto 3: la luz apagada

Con solo `Add(PA == 3).OnlyEnforceIf(b1)`, ¿se permite PA = 3 y b1 = 0?
¿Qué línea falta si b1 debe representar exactamente «PA es 3»?

<details>
<summary>Ver respuesta</summary>

Sí se permite: con b1 = 0 esa línea no se activa.
Falta `model.Add(PA != 3).OnlyEnforceIf(b1.Not())`.
Al agregarla, PA = 3 obliga a b1 = 1.

</details>

### Reto 4: ¿todos los días siguen disponibles?

Con las dos líneas de reificación, ¿PA puede valer 2? ¿Cuánto debe valer b1?

<details>
<summary>Ver respuesta</summary>

Sí puede valer 2; b1 debe valer 0. La reificación no elimina ese día.

</details>

### Reto 5: revisar las tres reglas

¿PA = 1 y PB = 1 es válido en la pregunta 4? ¿Y PA = 2 y PB = 2?

<details>
<summary>Ver respuesta</summary>

La primera pareja no: PB = 1 exige PA = 5.
La segunda sí: no se activa ninguna de las tres consecuencias.

</details>

### Reto 6: juntar tareas

Con capacidad 5, ¿pueden trabajar juntas tareas de demanda 3 y 2?
¿Y tres tareas de demanda 2?

<details>
<summary>Ver respuesta</summary>

3 + 2 = 5: sí. 2 + 2 + 2 = 6: no.
Comprobar solamente cada pareja no detecta el segundo problema.

</details>

### Reto 7: cambiar una regla y programarla

Tres niños deciden si participan en un juego. Deben participar **exactamente dos**
y Ana no puede participar. Crea tres booleanas y las dos restricciones.

<details>
<summary>Ver la idea antes del código</summary>

Suma de las tres luces igual a 2. La luz de Ana igual a 0.
Por tanto, Beto y Cami deben participar.

</details>

In [33]:
model = cp_model.CpModel()
ana = model.NewBoolVar("ana_participa")
beto = model.NewBoolVar("beto_participa")
cami = model.NewBoolVar("cami_participa")
model.Add(ana + beto + cami == 2)
model.Add(ana == 0)

solver, status = resolver(model)
if status in HAY_SOLUCION:
    print("Ana:", solver.Value(ana))
    print("Beto:", solver.Value(beto))
    print("Cami:", solver.Value(cami))

Estado: OPTIMAL
Ana: 0
Beto: 1
Cami: 1


### Reto 8: completa la equivalencia de una nota

La nota va de 0 a 20. `aprobado` debe valer 1 exactamente cuando la nota sea
al menos 11. ¿Cuál es la negación exacta de `nota >= 11`?

<details>
<summary>Ver respuesta</summary>

Es `nota < 11`; para notas enteras también se puede escribir `nota <= 10`.
Usamos una línea por cada caso, como con PA.

</details>

In [34]:
for nota_ejemplo in [10, 11]:
    model = cp_model.CpModel()
    nota = model.NewIntVar(0, 20, "nota")
    aprobado = model.NewBoolVar("aprobado")
    model.Add(nota >= 11).OnlyEnforceIf(aprobado)
    model.Add(nota < 11).OnlyEnforceIf(aprobado.Not())
    model.Add(nota == nota_ejemplo)

    solver, status = resolver(model)
    if status in HAY_SOLUCION:
        print("Nota:", solver.Value(nota), "Aprobado:", solver.Value(aprobado))

Estado: OPTIMAL
Nota: 10 Aprobado: 0
Estado: OPTIMAL
Nota: 11 Aprobado: 1


### Una plantilla para tu próximo ejercicio

Antes de programar, completa estas frases:

1. «Tengo que decidir…» → variables.
2. «Cada decisión puede valer…» → dominios.
3. «Debe cumplirse…» → restricciones.
4. «Quiero una respuesta válida / la mejor según…» → satisfacción u objetivo.
5. «Para saber si la respuesta sirve, voy a comprobar…» → interpretar la solución.

```python
from ortools.sat.python import cp_model

model = cp_model.CpModel()
# Crear las variables con sus dominios.
# Agregar las reglas del enunciado.
# Agregar un objetivo si se pide el mejor resultado.

solver = cp_model.CpSolver()
status = solver.Solve(model)
if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    # Leer e imprimir con solver.Value(...).
    pass
else:
    print(solver.StatusName(status))
```

Si puedes explicar qué pregunta responde cada variable y por qué pusiste cada
regla, ya puedes empezar a traducir un enunciado a un CSP.

## Procedencia: qué se leyó y dónde aparece en el repaso

Las explicaciones y los ejemplos pequeños son propios de este cuaderno, a partir
de los notebooks del repositorio. Se leyeron las celdas de texto y código de los
nueve archivos. Los dos notebooks de semana 4 tienen el mismo contenido de celdas.
Los ejemplos se adaptaron para poder ejecutarse sin mapas, corpus externos ni
otros archivos de datos.

| Notebook leído | Ideas aprovechadas |
|---|---|
| [Evaluaciones/PC2/solucionario_modificado.ipynb](../../Evaluaciones/PC2/solucionario_modificado.ipynb) | Médicos, dos formulaciones de recursos y reificación; paradas 9–12 y 22–24. |
| [Evaluaciones/PC2/Solucionario_PC2.ipynb](../../Evaluaciones/PC2/Solucionario_PC2.ipynb) | CSP de turnos, booleanas de inicio y antecedentes; paradas 10, 12 y 24. |
| [Unidades/1_ Metodología de investigación en Ciencias de la Computación/sem 02/02GNU_Plot_desde_Colab.ipynb](../../Unidades/1_%20Metodología%20de%20investigación%20en%20Ciencias%20de%20la%20Computación/sem%2002/02GNU_Plot_desde_Colab.ipynb) | Gráficos de funciones y datos; se contextualiza en la parada 21. |
| [Unidades/2_ Constraint Programming/sem 03/01_26_2_TopicosCC_3.ipynb](../../Unidades/2_%20Constraint%20Programming/sem%2003/01_26_2_TopicosCC_3.ipynb) | CSP básico, criptograma, cuadrados, horarios y anagramas; paradas 1–4, 14 y 20. |
| [Unidades/2_ Constraint Programming/sem 04/2.desClase.26_2_TopicosCC_4.ipynb](../../Unidades/2_%20Constraint%20Programming/sem%2004/2.desClase.26_2_TopicosCC_4.ipynb) | Grafos, sumas de vecinos, enumeración y minimizar colores; paradas 19, 25 y 26. |
| [Unidades/2_ Constraint Programming/sem 04/2.og.26_2_TopicosCC_4.ipynb](../../Unidades/2_%20Constraint%20Programming/sem%2004/2.og.26_2_TopicosCC_4.ipynb) | Mismos ejemplos de semana 4; paradas 19, 25 y 26. |
| [Unidades/3_ Programación por Restricciones y Data Mining/sem_06/22_1_TopicosCC_6.ipynb](../../Unidades/3_%20Programación%20por%20Restricciones%20y%20Data%20Mining/sem_06/22_1_TopicosCC_6.ipynb) | Grupos, matrices, conteos y productos; paradas 13 y 15–17. |
| [Unidades/3_ Programación por Restricciones y Data Mining/sem_07/22_1_TopicosCC_7.ipynb](../../Unidades/3_%20Programación%20por%20Restricciones%20y%20Data%20Mining/sem_07/22_1_TopicosCC_7.ipynb) | Palabras, tablas de transiciones y rimas; paradas 18 y 21. |
| [Unidades/3_ Programación por Restricciones y Data Mining/sem_07/Repaso_OR_Tools_Paso_a_Paso.ipynb](../../Unidades/3_%20Programación%20por%20Restricciones%20y%20Data%20Mining/sem_07/Repaso_OR_Tools_Paso_a_Paso.ipynb) | ALO/AMO/EO, lógica, estados, tablas, callbacks y depuración; se distribuyen por el repaso. |

## Documentación oficial para seguir estudiando

Para contrastar la notación y consultar los métodos usados:

- [OR-Tools: restricciones condicionales y channeling](https://developers.google.com/optimization/cp/channeling).
- [OR-Tools: resolución con CP-SAT y estados del solver](https://developers.google.com/optimization/cp/cp_solver).
- [Referencia Python de CpModel](https://or-tools.github.io/docs/python/classortools_1_1sat_1_1python_1_1cp__model_1_1CpModel.html).

Consultadas el 14 de octubre de 2026. Los ejemplos están preparados para
OR-Tools **9.14.6206**, la versión fijada en la celda de instalación.